# Covariate balancing API walkthrough

This notebook demonstrates the new causal-agnostic `CovariateBalancer` API and the lower-level calibration function it wraps.

The core abstraction is source-to-target reweighting:

```python
bal = CovariateBalancer(method="quadratic")
bal.fit(source_X, target_X=target_X)
w_source = bal.transform(source_X)
```

Causal estimators then use this object one treatment arm at a time to build a Riesz representer.


In [1]:
import numpy as np
import pandas as pd

from aipyw import AIPyW, CovariateBalancer
from aipyw.weights import balancing_weights
from aipyw.dgp import dgp_discrete

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(123)


## A source-to-target balancing problem

Create two covariate samples with different means. We want weights on the source sample so its weighted moments match the target moments.


In [2]:
n_source, n_target, p = 500, 800, 4
source_X = rng.normal(loc=np.array([0.8, -0.4, 0.3, 0.0]), scale=1.0, size=(n_source, p))
target_X = rng.normal(loc=np.zeros(p), scale=1.0, size=(n_target, p))

pd.DataFrame(
    {
        "source_unweighted": source_X.mean(axis=0),
        "target": target_X.mean(axis=0),
    },
    index=[f"X{j}" for j in range(p)],
)


,source_unweighted,target
X0,0.802388,0.018565
X1,-0.426697,-0.080978
X2,0.391682,-0.004590
X3,-0.033495,0.063888


## High-level API: `CovariateBalancer`

The direct calibration backends are:

- `method="quadratic"`: L2 / minimum-distance weights, allowing clipped sparse-ish weights.
- `method="entropy"`: exponential tilting / entropy balancing.
- `method="balnet"`: Adelie-backed calibration-loss path, normalized into source weights.

All three expose `fit`, `transform`, and `fit_transform`, and are cloneable sklearn estimators.


In [3]:
rows = []
weights_by_method = {}
for method in ["quadratic", "entropy", "balnet"]:
    bal = CovariateBalancer(method=method, n_lambdas=30, min_ratio=1e-2, progress_bar=False)
    weights = bal.fit_transform(source_X, target_X=target_X)
    weights_by_method[method] = weights
    weighted_mean = (weights / weights.sum()) @ source_X
    rows.append(
        {
            "method": method,
            "sum_weights": weights.sum(),
            "max_abs_mean_gap": np.max(np.abs(weighted_mean - target_X.mean(axis=0))),
            "ess": weights.sum() ** 2 / np.sum(weights ** 2),
            "min_weight": weights.min(),
            "max_weight": weights.max(),
        }
    )

pd.DataFrame(rows)


,method,sum_weights,max_abs_mean_gap,ess,min_weight,max_weight
0,quadratic,1.0,4.091405e-10,248.553096,0.000000,0.009279
1,entropy,1.0,2.810044e-11,211.467639,0.000042,0.020088
2,balnet,1.0,7.838229e-03,216.284739,0.000044,0.019434


In [4]:
pd.DataFrame(
    {
        "target": target_X.mean(axis=0),
        **{f"source_weighted_{m}": (w / w.sum()) @ source_X for m, w in weights_by_method.items()},
    },
    index=[f"X{j}" for j in range(p)],
)


,target,source_weighted_quadratic,source_weighted_entropy,source_weighted_balnet
X0,0.018565,0.018565,0.018565,0.026403
X1,-0.080978,-0.080978,-0.080978,-0.088413
X2,-0.004590,-0.004590,-0.004590,0.002751
X3,0.063888,0.063888,0.063888,0.056510


## Low-level function: `balancing_weights`

The low-level function expects a design matrix `Z = [1, source_X - target_mean]` and returns a link function plus dual coefficients. The intercept estimating equation normalizes weights to sum to one; the remaining equations target zero covariate imbalance.


In [5]:
target_mean = target_X.mean(axis=0)
Z = np.c_[np.ones(source_X.shape[0]), source_X - target_mean]
weight_link, beta, status = balancing_weights(
    Z,
    objective="entropy",
    min_weight=0.0,
    max_weight=10.0,
    l2_norm=0,
)
w_low = weight_link(Z @ beta)

pd.Series(
    {
        "solver_status": status,
        "sum_weights": w_low.sum(),
        "max_abs_balance_equation": np.max(np.abs(Z.T @ w_low - np.r_[1.0, np.zeros(p)])),
        "max_abs_mean_gap": np.max(np.abs((w_low / w_low.sum()) @ source_X - target_mean)),
    }
)


solver_status               1.000000e+00
sum_weights                 1.000000e+00
max_abs_balance_equation    4.310425e-10
max_abs_mean_gap            2.151011e-10
dtype: float64

## Causal use: one balancer per treatment arm

`AIPyW(riesz_method="balancing")` now uses `CovariateBalancer` internally. For each treatment arm, it fits source `X[W == w]` to target `X`, fills weights only on that arm, and leaves the Riesz representer zero off-arm.


In [6]:
Y, W, X = dgp_discrete(n=5_000, p=4, treat_effects=np.array([0.0, 0.4, 0.5, 0.55]))
causal_rows = []
for bal_obj in ["quadratic", "entropy", "balnet"]:
    model = AIPyW(riesz_method="balancing", bal_obj=bal_obj)
    model.fit(X, W, Y)
    off_arm_max = [np.max(np.abs(model.a_x[W != k, k])) for k in range(model.K)]
    on_arm_sum = [model.a_x[W == k, k].sum() for k in range(model.K)]
    causal_rows.append(
        {
            "bal_obj": bal_obj,
            "max_off_arm_weight": np.max(off_arm_max),
            "min_on_arm_sum": np.min(on_arm_sum),
            "max_on_arm_sum": np.max(on_arm_sum),
            **{k: v["effect"] for k, v in model.summary().items()},
        }
    )

pd.DataFrame(causal_rows)


,bal_obj,max_off_arm_weight,min_on_arm_sum,max_on_arm_sum,1 vs 0,2 vs 0,3 vs 0,2 vs 1,3 vs 1,3 vs 2
0,quadratic,0.0,1.0,1.0,0.39392,0.49904,0.54253,0.10513,0.14861,0.04348
1,entropy,0.0,1.0,1.0,0.39392,0.49904,0.54253,0.10513,0.14861,0.04348
2,balnet,0.0,1.0,1.0,0.39392,0.49904,0.54253,0.10513,0.14861,0.04348


## Takeaways

- `CovariateBalancer` is the generic source-to-target object; it is not intrinsically causal.
- `balancing_weights` is the low-level direct calibration equation solver.
- `AIPyW` uses the generic object to construct arm-specific Riesz representers.
- The balnet backend fits a regularized path through Adelie, while quadratic/entropy are one-shot calibration solves.
